# Chapter 5.10: SGLang Multi-Modal Support — Vision-Language Models

This notebook explores how SGLang handles multi-modal (vision-language) model serving. We cover the full pipeline from image preprocessing through patch extraction to token embedding, and examine how image data interacts with the KV cache.

## Learning Objectives
1. Understand multi-modal architecture: vision encoder + LLM
2. Implement the image preprocessing pipeline from scratch
3. Understand how images become token embeddings
4. Explore supported models (LLaVA, Qwen-VL, InternVL)
5. Analyze image caching and KV-cache interaction

## Prerequisites
- Basic understanding of CNNs and Vision Transformers (ViT)
- Familiarity with transformer architectures
- Python, PyTorch, numpy basics

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hideak1/vllm_study/blob/main/notebooks/part5/chapter_5.10_multimodal.ipynb)
[![Download Notebook](https://img.shields.io/badge/Download-Notebook-blue)](https://raw.githubusercontent.com/hideak1/vllm_study/main/notebooks/part5/chapter_5.10_multimodal.ipynb)

**How to do the exercises:**
1. **Google Colab (Recommended)**: Click the "Open in Colab" badge above — you get your own copy with free GPU.
2. **Local Jupyter**: Clone the repo, run `./start.sh`, then open notebooks at `http://localhost:8888`.
3. **Exercises**: Look for cells marked with 🏋️ **Exercise** or **Assignment**. Fill in the `TODO` sections and run the cell to check your work.

> **Tip**: In Colab, the notebook is automatically copied to your Drive — your changes are saved there.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from typing import List, Dict, Optional, Tuple, Union
from dataclasses import dataclass
import time
import json
import math

print(f"PyTorch version: {torch.__version__}")
DEVICE = torch.device('cpu')  # CPU for demonstrations
print("Ready for multi-modal exploration.")

---
## Part 1: Multi-Modal Architecture Overview

A vision-language model (VLM) combines:
1. **Vision Encoder** (e.g., CLIP ViT, SigLIP) — converts images to feature vectors
2. **Projection Module** (e.g., MLP, cross-attention) — maps vision features to LLM embedding space
3. **Language Model** (e.g., Llama, Qwen) — processes combined text + image tokens

```
Image ──► [Vision Encoder] ──► [Projection] ──► Image Tokens
                                                     │
Text  ──► [Tokenizer] ──► [Embedding] ──► Text Tokens
                                               │
                              ┌─────────────────┘
                              ▼
              [Concat: Text + Image Tokens]
                              │
                              ▼
                    [LLM Decoder Layers]
                              │
                              ▼
                        Generated Text
```

### Model Variants

| Model | Vision Encoder | Projection | LLM Base | Image Tokens |
|-------|---------------|------------|----------|--------------|
| LLaVA-1.5 | CLIP ViT-L/14 | 2-layer MLP | Vicuna/Llama | 576 |
| LLaVA-NeXT | SigLIP | MLP + AnyRes | Llama-3 | 576-2880 |
| Qwen-VL | ViT (custom) | Cross-attention | Qwen | 256 |
| InternVL-2 | InternViT-6B | MLP | InternLM2 | 256-1024 |

In [ ]:
# Simulate the multi-modal architecture components

@dataclass
class VLMConfig:
    """Configuration for a vision-language model."""
    # Vision encoder
    image_size: int = 336        # Input image resolution
    patch_size: int = 14         # ViT patch size
    vision_hidden_size: int = 1024  # Vision encoder hidden dim
    num_vision_layers: int = 24  # Vision encoder layers
    
    # Projection
    projection_type: str = 'mlp'  # 'mlp', 'cross_attention', 'linear'
    
    # Language model
    llm_hidden_size: int = 4096  # LLM hidden dim
    vocab_size: int = 32000
    max_position_embeddings: int = 4096
    
    @property
    def num_patches(self) -> int:
        """Number of image patches = (image_size / patch_size)^2."""
        return (self.image_size // self.patch_size) ** 2
    
    @property
    def num_image_tokens(self) -> int:
        """Number of tokens the image occupies in the LLM."""
        return self.num_patches  # Some models add a CLS token


# Common configurations
configs = {
    'LLaVA-1.5-7B': VLMConfig(image_size=336, patch_size=14, vision_hidden_size=1024, llm_hidden_size=4096),
    'LLaVA-NeXT-8B': VLMConfig(image_size=672, patch_size=14, vision_hidden_size=1024, llm_hidden_size=4096),
    'Qwen-VL-Chat': VLMConfig(image_size=448, patch_size=14, vision_hidden_size=1664, llm_hidden_size=4096),
    'InternVL2-8B': VLMConfig(image_size=448, patch_size=14, vision_hidden_size=3200, llm_hidden_size=4096),
}

print(f"{'Model':<20} {'Image Size':<12} {'Patches':<10} {'Image Tokens':<14} {'Vision dim':<12} {'LLM dim':<10}")
print("=" * 78)
for name, cfg in configs.items():
    print(f"{name:<20} {cfg.image_size:<12} {cfg.num_patches:<10} {cfg.num_image_tokens:<14} {cfg.vision_hidden_size:<12} {cfg.llm_hidden_size:<10}")

print(f"\nKey insight: A single image can occupy {configs['LLaVA-NeXT-8B'].num_image_tokens} tokens!")
print(f"That's equivalent to ~{configs['LLaVA-NeXT-8B'].num_image_tokens * 4} characters of text.")

---
## Part 2: Image Preprocessing Pipeline

Before an image can be fed to the vision encoder, it must be preprocessed:
1. **Resize** to the expected resolution
2. **Center crop** (or pad) to exact dimensions
3. **Normalize** pixel values (mean/std from training data)
4. **Convert** to tensor format (C, H, W)

Different models use different preprocessing!

In [ ]:
class ImagePreprocessor:
    """Image preprocessing pipeline for vision-language models.
    
    This mirrors the preprocessing in SGLang's image processing module.
    In production, this typically uses the model's processor from HuggingFace.
    """
    # ImageNet normalization (used by CLIP, SigLIP)
    IMAGENET_MEAN = [0.48145466, 0.4578275, 0.40821073]
    IMAGENET_STD = [0.26862954, 0.26130258, 0.27577711]
    
    # OpenAI CLIP normalization
    CLIP_MEAN = [0.48145466, 0.4578275, 0.40821073]
    CLIP_STD = [0.26862954, 0.26130258, 0.27577711]
    
    def __init__(
        self,
        image_size: int = 336,
        mean: List[float] = None,
        std: List[float] = None,
        resize_mode: str = 'shortest_edge',  # 'shortest_edge', 'squash', 'pad'
    ):
        self.image_size = image_size
        self.mean = np.array(mean or self.IMAGENET_MEAN)
        self.std = np.array(std or self.IMAGENET_STD)
        self.resize_mode = resize_mode
    
    def resize(self, image: np.ndarray) -> np.ndarray:
        """Resize image to target size."""
        h, w = image.shape[:2]
        target = self.image_size
        
        if self.resize_mode == 'squash':
            # Simple resize to exact dimensions
            from scipy.ndimage import zoom
            factors = (target / h, target / w, 1)
            return zoom(image, factors, order=1)
        
        elif self.resize_mode == 'shortest_edge':
            # Resize so shortest edge = target, then center crop
            scale = target / min(h, w)
            new_h, new_w = int(h * scale), int(w * scale)
            # Simple nearest-neighbor resize for demo
            indices_h = np.linspace(0, h - 1, new_h).astype(int)
            indices_w = np.linspace(0, w - 1, new_w).astype(int)
            resized = image[np.ix_(indices_h, indices_w)]
            return resized
        
        elif self.resize_mode == 'pad':
            # Resize to fit, pad to square
            scale = target / max(h, w)
            new_h, new_w = int(h * scale), int(w * scale)
            indices_h = np.linspace(0, h - 1, new_h).astype(int)
            indices_w = np.linspace(0, w - 1, new_w).astype(int)
            resized = image[np.ix_(indices_h, indices_w)]
            # Pad to square
            padded = np.zeros((target, target, 3), dtype=np.float32)
            pad_h = (target - new_h) // 2
            pad_w = (target - new_w) // 2
            padded[pad_h:pad_h+new_h, pad_w:pad_w+new_w] = resized
            return padded
        
        return image
    
    def center_crop(self, image: np.ndarray) -> np.ndarray:
        """Center crop to image_size x image_size."""
        h, w = image.shape[:2]
        target = self.image_size
        if h < target or w < target:
            return image
        top = (h - target) // 2
        left = (w - target) // 2
        return image[top:top+target, left:left+target]
    
    def normalize(self, image: np.ndarray) -> np.ndarray:
        """Normalize pixel values using mean and std."""
        img = image.astype(np.float32)
        if img.max() > 1.0:
            img = img / 255.0
        return (img - self.mean) / self.std
    
    def to_tensor(self, image: np.ndarray) -> torch.Tensor:
        """Convert (H, W, C) numpy array to (C, H, W) tensor."""
        return torch.from_numpy(image.transpose(2, 0, 1)).float()
    
    def preprocess(self, image: np.ndarray) -> torch.Tensor:
        """Full preprocessing pipeline."""
        # Step 1: Resize
        img = self.resize(image)
        # Step 2: Center crop
        img = self.center_crop(img)
        # Step 3: Normalize
        img = self.normalize(img)
        # Step 4: To tensor
        return self.to_tensor(img)


# Create a synthetic test image
def create_test_image(height: int, width: int, pattern: str = 'gradient') -> np.ndarray:
    """Create a synthetic test image."""
    if pattern == 'gradient':
        h_grad = np.linspace(0, 1, height)[:, None, None] * np.ones((1, width, 3))
        w_grad = np.linspace(0, 1, width)[None, :, None] * np.ones((height, 1, 3))
        img = (h_grad * 0.5 + w_grad * 0.5) * 255
    elif pattern == 'checkerboard':
        block = 32
        h_idx = np.arange(height) // block
        w_idx = np.arange(width) // block
        checker = (h_idx[:, None] + w_idx[None, :]) % 2
        img = np.stack([checker * 255, checker * 200, checker * 150], axis=-1)
    elif pattern == 'random':
        np.random.seed(42)
        img = np.random.randint(0, 256, (height, width, 3))
    else:
        img = np.ones((height, width, 3)) * 128
    return img.astype(np.float32)


# Demo: preprocess an image
preprocessor = ImagePreprocessor(image_size=336, resize_mode='shortest_edge')
test_img = create_test_image(480, 640, 'gradient')

print(f"Original image: {test_img.shape} (H, W, C)")
tensor = preprocessor.preprocess(test_img)
print(f"After preprocessing: {tensor.shape} (C, H, W)")
print(f"Value range: [{tensor.min():.2f}, {tensor.max():.2f}]")
print(f"Mean per channel: {tensor.mean(dim=(1,2)).tolist()}")

In [ ]:
# Visualize preprocessing steps
test_img = create_test_image(480, 640, 'checkerboard')

fig, axes = plt.subplots(2, 3, figsize=(15, 10))

# Original
axes[0, 0].imshow(test_img.astype(np.uint8))
axes[0, 0].set_title(f'Original ({test_img.shape[0]}x{test_img.shape[1]})')

# Different resize modes
for i, mode in enumerate(['shortest_edge', 'pad']):
    pp = ImagePreprocessor(image_size=336, resize_mode=mode)
    resized = pp.resize(test_img)
    cropped = pp.center_crop(resized) if mode == 'shortest_edge' else resized
    display = np.clip(cropped / 255.0 if cropped.max() > 1 else cropped, 0, 1)
    axes[0, i+1].imshow(display)
    axes[0, i+1].set_title(f'{mode} ({cropped.shape[0]}x{cropped.shape[1]})')

# Show normalized
pp = ImagePreprocessor(image_size=336)
normalized = pp.normalize(pp.center_crop(pp.resize(test_img)))
# Unnormalize for display
display_norm = normalized * pp.std + pp.mean
display_norm = np.clip(display_norm, 0, 1)
axes[1, 0].imshow(display_norm)
axes[1, 0].set_title('After Normalization')

# Show per-channel histograms
tensor = pp.preprocess(test_img)
for ch, color in enumerate(['red', 'green', 'blue']):
    axes[1, 1].hist(tensor[ch].flatten().numpy(), bins=50, alpha=0.5, color=color, label=f'Ch {ch}')
axes[1, 1].set_title('Pixel Distribution After Preprocessing')
axes[1, 1].legend()

# Patch visualization
patch_size = 14
n_patches_h = 336 // patch_size
n_patches_w = 336 // patch_size
patch_grid = np.zeros((336, 336, 3))
for i in range(n_patches_h):
    for j in range(n_patches_w):
        color = plt.cm.tab20((i * n_patches_w + j) % 20)[:3]
        patch_grid[i*patch_size:(i+1)*patch_size, j*patch_size:(j+1)*patch_size] = color
axes[1, 2].imshow(patch_grid)
axes[1, 2].set_title(f'Patch Grid ({n_patches_h}x{n_patches_w} = {n_patches_h*n_patches_w} patches)')

for ax in axes.flat:
    ax.axis('off') if ax != axes[1, 1] else None

plt.suptitle('Image Preprocessing Pipeline', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

---
## Part 3: How Images Become Token Embeddings

The vision encoder (typically a ViT) converts the image into a sequence of feature vectors, one per patch. These are then projected to match the LLM's embedding dimension.

In [ ]:
class PatchEmbedding(nn.Module):
    """Convert image patches to embeddings.
    
    This is the first layer of a ViT. It splits the image into patches
    and projects each patch through a linear layer.
    """
    def __init__(self, image_size: int = 336, patch_size: int = 14,
                 in_channels: int = 3, embed_dim: int = 1024):
        super().__init__()
        self.image_size = image_size
        self.patch_size = patch_size
        self.num_patches = (image_size // patch_size) ** 2
        
        # Conv2d with kernel_size=patch_size and stride=patch_size
        # This is equivalent to splitting into patches + linear projection
        self.proj = nn.Conv2d(
            in_channels, embed_dim,
            kernel_size=patch_size, stride=patch_size
        )
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """x: (B, C, H, W) -> (B, num_patches, embed_dim)"""
        # Conv2d: (B, C, H, W) -> (B, embed_dim, H/patch, W/patch)
        x = self.proj(x)
        # Flatten spatial dims and transpose: -> (B, num_patches, embed_dim)
        x = x.flatten(2).transpose(1, 2)
        return x


class VisionProjector(nn.Module):
    """Project vision features to LLM embedding space.
    
    LLaVA uses a 2-layer MLP:
    vision_dim -> 4*vision_dim -> llm_dim
    """
    def __init__(self, vision_dim: int, llm_dim: int, projection_type: str = 'mlp'):
        super().__init__()
        if projection_type == 'mlp':
            self.proj = nn.Sequential(
                nn.Linear(vision_dim, vision_dim * 4),
                nn.GELU(),
                nn.Linear(vision_dim * 4, llm_dim),
            )
        elif projection_type == 'linear':
            self.proj = nn.Linear(vision_dim, llm_dim)
        self.projection_type = projection_type
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.proj(x)


# Demo: full pipeline from image to LLM tokens
config = configs['LLaVA-1.5-7B']

# Step 1: Patch embedding
patch_embed = PatchEmbedding(
    image_size=config.image_size,
    patch_size=config.patch_size,
    embed_dim=config.vision_hidden_size
)

# Step 2: Projector
projector = VisionProjector(
    config.vision_hidden_size,
    config.llm_hidden_size,
    'mlp'
)

# Process a test image
pp = ImagePreprocessor(image_size=config.image_size)
test_img = create_test_image(480, 640, 'gradient')
img_tensor = pp.preprocess(test_img).unsqueeze(0)  # Add batch dim

print("=== Image to Token Pipeline ===")
print(f"1. Raw image:        {test_img.shape}")
print(f"2. Preprocessed:     {img_tensor.shape}")

patch_embeddings = patch_embed(img_tensor)
print(f"3. Patch embeddings: {patch_embeddings.shape}")

image_tokens = projector(patch_embeddings)
print(f"4. Image tokens:     {image_tokens.shape}")
print(f"\nThe image is now {image_tokens.shape[1]} tokens in the LLM's embedding space!")
print(f"Each token has dimension {image_tokens.shape[2]} (same as text tokens).")

In [ ]:
# Visualize: how text and image tokens are combined

def visualize_token_sequence(
    text_before: str = "<image>\nDescribe this image.",
    num_image_tokens: int = 576,
    text_after: str = "The image shows"
):
    """Visualize how a multi-modal prompt is tokenized."""
    # Simulate tokenization
    text_tokens_before = text_before.split()
    text_tokens_after = text_after.split()
    
    fig, ax = plt.subplots(figsize=(16, 3))
    
    # System prompt tokens
    x_pos = 0
    for i, tok in enumerate(text_tokens_before):
        if tok == '<image>':
            continue
        ax.barh(0, 1, left=x_pos, color='lightblue', edgecolor='black', linewidth=0.5)
        ax.text(x_pos + 0.5, 0, tok, ha='center', va='center', fontsize=6)
        x_pos += 1
    
    # Image tokens (compressed visualization)
    img_start = x_pos
    ax.barh(0, num_image_tokens * 0.02, left=x_pos, color='coral', 
            edgecolor='black', linewidth=0.5, height=0.8)
    ax.text(x_pos + num_image_tokens * 0.01, 0, 
            f'[{num_image_tokens} image tokens]', 
            ha='center', va='center', fontsize=8, fontweight='bold')
    x_pos += num_image_tokens * 0.02
    
    # After-image text tokens
    for tok in text_tokens_after:
        ax.barh(0, 1, left=x_pos, color='lightgreen', edgecolor='black', linewidth=0.5)
        ax.text(x_pos + 0.5, 0, tok, ha='center', va='center', fontsize=6)
        x_pos += 1
    
    # Legend
    patches = [
        mpatches.Patch(color='lightblue', label='Text tokens (prompt)'),
        mpatches.Patch(color='coral', label=f'Image tokens ({num_image_tokens})'),
        mpatches.Patch(color='lightgreen', label='Text tokens (generation)'),
    ]
    ax.legend(handles=patches, loc='upper right')
    
    ax.set_xlim(-0.5, x_pos + 1)
    ax.set_ylim(-1, 1)
    ax.set_xlabel('Sequence Position')
    ax.set_title('Multi-Modal Token Sequence Layout')
    ax.set_yticks([])
    
    plt.tight_layout()
    plt.show()
    
    total = len([t for t in text_tokens_before if t != '<image>']) + num_image_tokens + len(text_tokens_after)
    print(f"Total sequence length: {total} tokens")
    print(f"Image tokens are {num_image_tokens/total*100:.1f}% of the sequence")


visualize_token_sequence()

---
## Part 4: Simplified Vision Encoder (ViT)

The vision encoder is a standard Vision Transformer (ViT). Let's build a minimal version.

In [ ]:
class SimplifiedViT(nn.Module):
    """Simplified Vision Transformer for demonstration.
    
    Real implementations (CLIP ViT, SigLIP) have more details
    like pre/post LayerNorm, class token, etc.
    """
    def __init__(self, config: VLMConfig):
        super().__init__()
        self.config = config
        
        # Patch embedding
        self.patch_embed = PatchEmbedding(
            config.image_size, config.patch_size,
            embed_dim=config.vision_hidden_size
        )
        
        # Position embeddings
        num_patches = config.num_patches
        self.pos_embed = nn.Parameter(
            torch.randn(1, num_patches, config.vision_hidden_size) * 0.02
        )
        
        # Simplified transformer layers (just self-attention + FFN)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=config.vision_hidden_size,
            nhead=16,
            dim_feedforward=config.vision_hidden_size * 4,
            batch_first=True,
            norm_first=True
        )
        # Use fewer layers for demo
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=2)
        self.norm = nn.LayerNorm(config.vision_hidden_size)
    
    def forward(self, pixel_values: torch.Tensor) -> torch.Tensor:
        """pixel_values: (B, C, H, W) -> (B, num_patches, vision_hidden_size)"""
        # Step 1: Patch embedding
        x = self.patch_embed(pixel_values)
        
        # Step 2: Add position embeddings
        x = x + self.pos_embed
        
        # Step 3: Transformer encoder
        x = self.encoder(x)
        
        # Step 4: LayerNorm
        x = self.norm(x)
        
        return x


# Demo: full vision pipeline
config = configs['LLaVA-1.5-7B']
vit = SimplifiedViT(config)
projector = VisionProjector(config.vision_hidden_size, config.llm_hidden_size)

# Count parameters
vit_params = sum(p.numel() for p in vit.parameters()) / 1e6
proj_params = sum(p.numel() for p in projector.parameters()) / 1e6

print(f"Vision Encoder (simplified): {vit_params:.1f}M parameters")
print(f"Projector: {proj_params:.1f}M parameters")

# Forward pass
img_tensor = pp.preprocess(test_img).unsqueeze(0)
with torch.no_grad():
    start = time.time()
    vision_features = vit(img_tensor)
    image_tokens = projector(vision_features)
    elapsed = (time.time() - start) * 1000

print(f"\nForward pass:")
print(f"  Input:           {img_tensor.shape}")
print(f"  Vision features: {vision_features.shape}")
print(f"  Image tokens:    {image_tokens.shape}")
print(f"  Time:            {elapsed:.1f} ms")

---
## Part 5: Image Caching and KV-Cache Interaction

Image processing is expensive. SGLang optimizes by:
1. **Caching** preprocessed image embeddings
2. **Sharing** image KV-cache across requests with the same image
3. **Prefix caching** treats image tokens as part of the prefix

In [ ]:
class ImageCache:
    """Cache for processed image embeddings.
    
    In SGLang, this is integrated with RadixAttention's prefix cache.
    Images that appear in multiple requests only need to be encoded once.
    """
    def __init__(self, max_size: int = 100):
        self.cache: Dict[str, torch.Tensor] = {}
        self.access_order: List[str] = []
        self.max_size = max_size
        self.stats = {'hits': 0, 'misses': 0}
    
    def _compute_key(self, image: np.ndarray) -> str:
        """Compute a hash key for an image."""
        import hashlib
        return hashlib.md5(image.tobytes()).hexdigest()
    
    def get(self, image: np.ndarray) -> Optional[torch.Tensor]:
        key = self._compute_key(image)
        if key in self.cache:
            self.stats['hits'] += 1
            self.access_order.remove(key)
            self.access_order.append(key)
            return self.cache[key]
        self.stats['misses'] += 1
        return None
    
    def put(self, image: np.ndarray, embeddings: torch.Tensor):
        key = self._compute_key(image)
        if len(self.cache) >= self.max_size:
            oldest = self.access_order.pop(0)
            del self.cache[oldest]
        self.cache[key] = embeddings
        self.access_order.append(key)
    
    @property
    def hit_rate(self) -> float:
        total = self.stats['hits'] + self.stats['misses']
        return self.stats['hits'] / total if total > 0 else 0.0


# Demo: image caching benefit
cache = ImageCache(max_size=50)

# Simulate traffic: many requests with same images
images = [create_test_image(480, 640, p) for p in ['gradient', 'checkerboard', 'random']]

# Traffic: 30 requests, reusing images
np.random.seed(42)
request_images = [images[np.random.randint(0, len(images))] for _ in range(30)]

encode_times = []
for img in request_images:
    cached = cache.get(img)
    if cached is not None:
        encode_times.append(0.0)  # Cache hit: no encoding needed
    else:
        # Simulate encoding time
        start = time.time()
        tensor = pp.preprocess(img).unsqueeze(0)
        with torch.no_grad():
            embeddings = projector(vit(tensor))
        elapsed = (time.time() - start) * 1000
        encode_times.append(elapsed)
        cache.put(img, embeddings)

print(f"=== Image Cache Performance ===")
print(f"Total requests: 30")
print(f"Cache hits: {cache.stats['hits']}")
print(f"Cache misses: {cache.stats['misses']}")
print(f"Hit rate: {cache.hit_rate:.1%}")
print(f"Avg encode time (cache miss): {np.mean([t for t in encode_times if t > 0]):.1f} ms")
print(f"Avg encode time (all): {np.mean(encode_times):.1f} ms")
print(f"Time saved by caching: {(1 - np.mean(encode_times) / np.mean([t for t in encode_times if t > 0])) * 100:.0f}%")

In [ ]:
# KV-cache analysis for multi-modal models

def analyze_kv_cache_impact(
    text_tokens: int,
    image_tokens: int,
    num_images: int = 1,
    num_layers: int = 32,
    num_kv_heads: int = 8,
    head_dim: int = 128,
    dtype_bytes: int = 2  # fp16
) -> dict:
    """Analyze KV-cache memory for multi-modal sequences."""
    total_tokens = text_tokens + image_tokens * num_images
    
    # KV cache: 2 (K and V) * num_layers * num_kv_heads * head_dim * dtype_bytes
    kv_per_token = 2 * num_layers * num_kv_heads * head_dim * dtype_bytes
    
    text_kv_bytes = text_tokens * kv_per_token
    image_kv_bytes = image_tokens * num_images * kv_per_token
    total_kv_bytes = total_tokens * kv_per_token
    
    return {
        'total_tokens': total_tokens,
        'text_tokens': text_tokens,
        'image_tokens': image_tokens * num_images,
        'text_kv_mb': text_kv_bytes / (1024 * 1024),
        'image_kv_mb': image_kv_bytes / (1024 * 1024),
        'total_kv_mb': total_kv_bytes / (1024 * 1024),
        'image_kv_fraction': image_kv_bytes / total_kv_bytes
    }


# Compare scenarios
scenarios = [
    ("Text only (500 tokens)", 500, 0, 0),
    ("1 image (LLaVA-1.5)", 100, 576, 1),
    ("1 image (LLaVA-NeXT)", 100, 2304, 1),
    ("3 images (LLaVA-1.5)", 200, 576, 3),
    ("5 images (high-res)", 200, 2304, 5),
]

print(f"{'Scenario':<30} {'Total Tok':<12} {'Text KV (MB)':<14} {'Image KV (MB)':<14} {'Total KV (MB)':<14} {'Image %':<10}")
print("=" * 94)
for name, text, img_tok, n_img in scenarios:
    result = analyze_kv_cache_impact(text, img_tok, n_img)
    print(f"{name:<30} {result['total_tokens']:<12} {result['text_kv_mb']:<14.2f} {result['image_kv_mb']:<14.2f} {result['total_kv_mb']:<14.2f} {result['image_kv_fraction']*100:<10.1f}")

print(f"\nKey insight: Images can dominate KV-cache memory!")
print(f"Image caching and prefix sharing are critical for multi-modal serving.")

---
## Part 6: Multi-Image and AnyRes Support

Modern VLMs support multiple images and dynamic resolutions:

In [ ]:
class AnyResPreprocessor:
    """Any-Resolution preprocessing (used by LLaVA-NeXT, InternVL).
    
    Instead of resizing all images to one fixed size, this:
    1. Finds the best-fitting grid of patches
    2. Splits the image into tiles
    3. Also includes a downscaled overview image
    
    This preserves detail in high-resolution images.
    """
    def __init__(self, base_size: int = 336, max_tiles: int = 6):
        self.base_size = base_size
        self.max_tiles = max_tiles
        # Possible grid configurations
        self.grid_configs = self._generate_grids()
    
    def _generate_grids(self) -> List[Tuple[int, int]]:
        """Generate valid grid configurations."""
        grids = []
        for h in range(1, self.max_tiles + 1):
            for w in range(1, self.max_tiles + 1):
                if h * w <= self.max_tiles:
                    grids.append((h, w))
        return grids
    
    def find_best_grid(self, image_h: int, image_w: int) -> Tuple[int, int]:
        """Find the grid that best matches the image aspect ratio."""
        aspect_ratio = image_w / image_h
        best_grid = (1, 1)
        best_score = float('inf')
        
        for gh, gw in self.grid_configs:
            grid_ratio = gw / gh
            score = abs(grid_ratio - aspect_ratio)
            # Prefer larger grids (more detail) with small penalty
            score -= 0.01 * gh * gw
            if score < best_score:
                best_score = score
                best_grid = (gh, gw)
        
        return best_grid
    
    def split_tiles(self, image: np.ndarray, grid: Tuple[int, int]) -> List[np.ndarray]:
        """Split image into tiles based on grid."""
        gh, gw = grid
        h, w = image.shape[:2]
        tile_h = h // gh
        tile_w = w // gw
        
        tiles = []
        for i in range(gh):
            for j in range(gw):
                tile = image[i*tile_h:(i+1)*tile_h, j*tile_w:(j+1)*tile_w]
                tiles.append(tile)
        return tiles
    
    def preprocess(self, image: np.ndarray) -> dict:
        """Full AnyRes preprocessing."""
        h, w = image.shape[:2]
        grid = self.find_best_grid(h, w)
        
        # Get tiles
        tiles = self.split_tiles(image, grid)
        
        # Add overview (downscaled full image)
        pp = ImagePreprocessor(image_size=self.base_size, resize_mode='pad')
        overview = pp.resize(image)
        
        num_patches_per_tile = (self.base_size // 14) ** 2
        total_tokens = (len(tiles) + 1) * num_patches_per_tile  # +1 for overview
        
        return {
            'grid': grid,
            'num_tiles': len(tiles),
            'total_images': len(tiles) + 1,
            'total_tokens': total_tokens,
            'overview_shape': overview.shape,
        }


# Demo: AnyRes for different image sizes
anyres = AnyResPreprocessor(base_size=336, max_tiles=6)

test_images = [
    ("Square 512x512", create_test_image(512, 512, 'gradient')),
    ("Wide 360x1280", create_test_image(360, 1280, 'checkerboard')),
    ("Tall 1024x384", create_test_image(1024, 384, 'gradient')),
    ("Large 1920x1080", create_test_image(1080, 1920, 'random')),
    ("Small 128x128", create_test_image(128, 128, 'checkerboard')),
]

print(f"{'Image':<20} {'Size':<15} {'Grid':<10} {'Tiles':<8} {'Total Tokens':<14}")
print("=" * 67)
for name, img in test_images:
    result = anyres.preprocess(img)
    print(f"{name:<20} {f'{img.shape[0]}x{img.shape[1]}':<15} {str(result['grid']):<10} {result['num_tiles']:<8} {result['total_tokens']:<14}")

print(f"\nWith AnyRes, a wide panorama gets more tiles (more detail) than a small thumbnail.")

In [ ]:
# Multi-image support analysis

class MultiImageManager:
    """Manages multiple images in a single request."""
    
    def __init__(self, max_images: int = 8, max_total_tokens: int = 4096):
        self.max_images = max_images
        self.max_total_tokens = max_total_tokens
    
    def plan_layout(
        self,
        images: List[np.ndarray],
        text_tokens: int,
        tokens_per_image: int = 576
    ) -> dict:
        """Plan how to fit multiple images + text within token budget."""
        n_images = min(len(images), self.max_images)
        total_image_tokens = n_images * tokens_per_image
        total_tokens = text_tokens + total_image_tokens
        
        fits = total_tokens <= self.max_total_tokens
        
        return {
            'num_images': n_images,
            'text_tokens': text_tokens,
            'image_tokens': total_image_tokens,
            'total_tokens': total_tokens,
            'fits_context': fits,
            'remaining_budget': max(0, self.max_total_tokens - total_tokens),
            'max_response_tokens': max(0, self.max_total_tokens - text_tokens - total_image_tokens)
        }


# Demo
mm = MultiImageManager(max_images=8, max_total_tokens=4096)

scenarios_mi = [
    ("Single image chat", 1, 100),
    ("Compare 2 images", 2, 150),
    ("Document with 4 images", 4, 500),
    ("Image gallery (8 images)", 8, 200),
]

print(f"{'Scenario':<30} {'Images':<8} {'Text Tok':<10} {'Img Tok':<10} {'Total':<8} {'Fits?':<6} {'Max Response':<12}")
print("=" * 84)
for name, n_imgs, text_tok in scenarios_mi:
    imgs = [create_test_image(336, 336, 'random') for _ in range(n_imgs)]
    result = mm.plan_layout(imgs, text_tok)
    status = 'Yes' if result['fits_context'] else 'NO'
    print(f"{name:<30} {result['num_images']:<8} {result['text_tokens']:<10} {result['image_tokens']:<10} {result['total_tokens']:<8} {status:<6} {result['max_response_tokens']:<12}")

---
## Part 7: SGLang Multi-Modal Integration

SGLang integrates multi-modal support through its runtime.

In [ ]:
# SGLang multi-modal API usage examples

sglang_examples = {
    "Launch VLM server": {
        "command": (
            "python -m sglang.launch_server "
            "--model-path lmms-lab/llava-onevision-qwen2-7b-ov "
            "--chat-template chatml-llava "
            "--port 30000"
        ),
        "description": "Start SGLang server with LLaVA-OneVision model"
    },
    "Single image (OpenAI API)": {
        "code": '''
from openai import OpenAI
client = OpenAI(base_url="http://localhost:30000/v1", api_key="none")

response = client.chat.completions.create(
    model="default",
    messages=[{
        "role": "user",
        "content": [
            {"type": "text", "text": "Describe this image in detail."},
            {"type": "image_url", "image_url": {"url": "https://example.com/photo.jpg"}}
        ]
    }],
    max_tokens=300
)
print(response.choices[0].message.content)
''',
        "description": "Use OpenAI-compatible API for single image"
    },
    "Multi-image (SGLang native)": {
        "code": '''
import sglang as sgl

@sgl.function
def compare_images(s, image1, image2):
    s += sgl.user(
        sgl.image(image1) + sgl.image(image2) +
        "Compare these two images. What are the main differences?"
    )
    s += sgl.assistant(sgl.gen("comparison", max_tokens=500))

state = compare_images.run(
    image1="photo1.jpg",
    image2="photo2.jpg"
)
print(state["comparison"])
''',
        "description": "SGLang native API for multi-image"
    },
}

for name, info in sglang_examples.items():
    print(f"=== {name} ===")
    print(f"Description: {info['description']}")
    if 'command' in info:
        print(f"Command: {info['command']}")
    if 'code' in info:
        print(info['code'])
    print()

In [ ]:
# Profile: image encoding overhead breakdown

def profile_image_pipeline(image: np.ndarray, config: VLMConfig, num_runs: int = 20):
    """Profile each stage of the image encoding pipeline."""
    pp = ImagePreprocessor(image_size=config.image_size)
    vit_model = SimplifiedViT(config)
    proj = VisionProjector(config.vision_hidden_size, config.llm_hidden_size)
    
    timings = {'preprocess': [], 'patch_embed': [], 'vit_encode': [], 'project': []}
    
    for _ in range(num_runs):
        # Preprocessing
        t0 = time.time()
        tensor = pp.preprocess(image).unsqueeze(0)
        timings['preprocess'].append((time.time() - t0) * 1000)
        
        with torch.no_grad():
            # Patch embedding
            t0 = time.time()
            patches = vit_model.patch_embed(tensor)
            timings['patch_embed'].append((time.time() - t0) * 1000)
            
            # ViT encoding
            t0 = time.time()
            features = vit_model.encoder(patches + vit_model.pos_embed)
            features = vit_model.norm(features)
            timings['vit_encode'].append((time.time() - t0) * 1000)
            
            # Projection
            t0 = time.time()
            tokens = proj(features)
            timings['project'].append((time.time() - t0) * 1000)
    
    avg_timings = {k: np.mean(v) for k, v in timings.items()}
    total = sum(avg_timings.values())
    
    return avg_timings, total


# Profile for LLaVA-1.5 config
test_img = create_test_image(640, 480, 'random')
config = configs['LLaVA-1.5-7B']
timings, total = profile_image_pipeline(test_img, config)

print(f"=== Image Encoding Pipeline Profile ({config.image_size}px) ===")
print(f"{'Stage':<20} {'Time (ms)':<12} {'Fraction':<10}")
print("-" * 42)
for stage, ms in timings.items():
    print(f"{stage:<20} {ms:<12.2f} {ms/total*100:<10.1f}%")
print(f"{'TOTAL':<20} {total:<12.2f}")

# Visualization
fig, ax = plt.subplots(figsize=(10, 5))
stages = list(timings.keys())
times = list(timings.values())
colors = ['#3498db', '#2ecc71', '#e74c3c', '#f39c12']
ax.barh(stages, times, color=colors)
ax.set_xlabel('Time (ms)')
ax.set_title('Image Encoding Pipeline Breakdown')
for i, t in enumerate(times):
    ax.text(t + 0.1, i, f'{t:.1f}ms ({t/total*100:.0f}%)', va='center')
plt.tight_layout()
plt.show()

---
## Hands-On Assignments

### Assignment 1: Implement an Image Preprocessing Pipeline

**Task**: Build a complete, configurable image preprocessing pipeline.

Requirements:
1. Support multiple resize modes: `crop`, `pad`, `stretch`, `anyres`
2. Support different normalization presets (CLIP, ImageNet, SigLIP)
3. Implement data augmentation options (for benchmarking robustness)
4. Add batch processing support
5. Compare preprocessing outputs across modes visually

In [ ]:
# === ASSIGNMENT 1: Full Image Preprocessing Pipeline ===

class FullPreprocessor:
    """
    TODO: Implement a production-grade image preprocessing pipeline.
    
    Features:
    - Multiple resize modes
    - Normalization presets
    - Optional augmentation (flip, rotate, color jitter)
    - Batch processing
    - Pipeline profiling
    """
    def __init__(self, config: dict):
        # YOUR CODE HERE
        pass
    
    def preprocess_single(self, image: np.ndarray) -> torch.Tensor:
        # YOUR CODE HERE
        pass
    
    def preprocess_batch(self, images: List[np.ndarray]) -> torch.Tensor:
        # YOUR CODE HERE
        pass

print("Assignment 1: Build FullPreprocessor with multiple modes and batch support.")

### Assignment 2: Build a Visual QA Demo Application

**Task**: Build a mock Visual QA system that simulates multi-modal inference.

Requirements:
1. Accept image + question pairs
2. Implement the full pipeline: preprocess -> encode -> project -> concat with text -> "generate"
3. Support multi-image inputs
4. Track and report token usage (image vs text)
5. Implement batch processing for multiple questions about the same image

In [ ]:
# === ASSIGNMENT 2: Visual QA Demo ===

class VisualQASystem:
    """
    TODO: Implement a visual question answering system.
    
    Features:
    - Image + question input
    - Full preprocessing -> encoding pipeline
    - Multi-image support
    - Token usage tracking
    - Batch processing for same-image queries
    """
    def __init__(self, config: VLMConfig):
        # YOUR CODE HERE
        pass
    
    def answer(self, image: np.ndarray, question: str) -> dict:
        # YOUR CODE HERE
        pass
    
    def batch_answer(self, image: np.ndarray, questions: List[str]) -> List[dict]:
        # YOUR CODE HERE
        pass

print("Assignment 2: Build VisualQASystem with multi-image and batch support.")

### Assignment 3: Profile Image Encoding Overhead

**Task**: Build a comprehensive profiling tool for the image encoding pipeline.

Requirements:
1. Profile each stage independently: preprocessing, patch embedding, ViT, projection
2. Measure memory consumption at each stage
3. Test with different image sizes and batch sizes
4. Compare with/without image caching
5. Generate a detailed report with charts

In [ ]:
# === ASSIGNMENT 3: Image Encoding Profiler ===

class ImagePipelineProfiler:
    """
    TODO: Comprehensive profiler for image encoding.
    
    Features:
    - Per-stage timing
    - Memory tracking
    - Multi-resolution comparison
    - Cache effectiveness analysis
    - Report generation
    """
    def __init__(self, config: VLMConfig):
        # YOUR CODE HERE
        pass
    
    def profile(self, images: List[np.ndarray], batch_sizes: List[int]) -> dict:
        # YOUR CODE HERE
        pass
    
    def report(self) -> str:
        # YOUR CODE HERE
        pass

print("Assignment 3: Build ImagePipelineProfiler and generate a performance report.")

---
## Summary

| Component | Purpose | Key Detail |
|-----------|---------|------------|
| Image Preprocessing | Resize, crop, normalize | Model-specific settings |
| Vision Encoder (ViT) | Image -> patch features | 576+ tokens per image |
| Projection (MLP) | Vision dim -> LLM dim | 2-layer MLP (LLaVA) |
| Token Concatenation | Mix image + text tokens | Image tokens = prefix |
| Image Cache | Avoid re-encoding | Hash-based, LRU eviction |
| AnyRes | Dynamic resolution | Tiles + overview image |

### Key Takeaways
1. Images are converted to token sequences that the LLM processes like text
2. A single image can use 576-2880 tokens — dominating context length and KV-cache
3. Image caching is critical: same image across requests only needs encoding once
4. AnyRes preserves detail in high-resolution images via tiling
5. Multi-image support requires careful token budget management

In [ ]:
print("=== End of Chapter 5.10: SGLang Multi-Modal Support ===")
print("\nSupported VLMs in SGLang:")
print("  - LLaVA-1.5, LLaVA-NeXT, LLaVA-OneVision")
print("  - Qwen-VL, Qwen2-VL")
print("  - InternVL, InternVL2")
print("  - Llama 3.2 Vision")
print("\nLaunch: python -m sglang.launch_server --model-path <vlm-path> --chat-template <template>")